In [2]:
import pandas as pd

df = pd.read_csv('dataset\customer_shopping_behavior.csv')
df.head(2)

<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\ibyir\AppData\Local\Temp\ipykernel_26792\2419149453.py:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  df = pd.read_csv('dataset\customer_shopping_behavior.csv')


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly


In [3]:
df.isnull().sum()
#to replace null values with median. but we will use median of each
#category. could we use median of whole column? no, because it will be biased by the category with more data. so we will use median of each category

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [4]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [6]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [5]:
#One issue is column names. we can snake case them
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.rename(columns= {'purchase_amount_(usd)': 'purchase_amount'}, inplace=True)
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [6]:
#group customers into young adult, adult, middle age, and senior
labels = ['Young Adult', 'Adult', 'Middle Age', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle Age
1,19,Young Adult
2,50,Middle Age
3,21,Young Adult
4,45,Middle Age
5,46,Middle Age
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle Age


In [7]:
#creating another feature called purchase_frequency_days
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-weekly': 14,
    'Annually': 365, 
    'Every 3 Months': 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14.0
1,Fortnightly,14.0
2,Weekly,7.0
3,Weekly,7.0
4,Annually,365.0
5,Weekly,7.0
6,Quarterly,90.0
7,Weekly,7.0
8,Annually,365.0
9,Quarterly,90.0


In [8]:
#is discount_applied and promo_code_used the same? we can check if 
#it's not redunancy by checking if they are the same for all rows
(df['discount_applied'] == df['promo_code_used']).all()
#let's drop promo_code_used
df = df.drop(columns=['promo_code_used'], axis=1)
df

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle Age,14.0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14.0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle Age,7.0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7.0
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle Age,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,32,Venmo,Weekly,Adult,7.0
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,41,Bank Transfer,Bi-Weekly,Middle Age,NaN
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,24,Venmo,Quarterly,Middle Age,90.0
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,24,Venmo,Weekly,Adult,7.0


In [40]:
df.dtypes

customer_id                   int64
age                           int64
gender                       object
item_purchased               object
category                     object
purchase_amount               int64
location                     object
size                         object
color                        object
season                       object
review_rating               float64
subscription_status          object
shipping_type                object
discount_applied             object
previous_purchases            int64
payment_method               object
frequency_of_purchases       object
age_group                  category
purchase_frequency_days     float64
dtype: object

In [9]:
df['gender'].unique()
df['subscription_status'].unique()
df['discount_applied'].unique()
df.describe()

,customer_id,age,purchase_amount,review_rating,previous_purchases,purchase_frequency_days
count,3900.000000,3900.000000,3900.000000,3900.000000,3900.000000,3353.000000
mean,1950.500000,44.068462,59.764359,3.750051,25.351538,101.390098
std,1125.977353,15.207589,23.685392,0.713590,14.447125,124.140318
min,1.000000,18.000000,20.000000,2.500000,1.000000,7.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000,14.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000,90.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000,90.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000,365.000000


In [2]:
pip install sqlalchemy

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.1 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 7.1 MB/s  0:00:00

   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   ---------------------------------------- 2/2 [sqlalchemy]

Note: you may need to restart the kernel to use 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
#then we export this cleaned data to a postgresql database for sql analysis
#we need to connect with db.
from sqlalchemy import create_engine

username = 'postgres'
password = 'mydbmydb'
host = 'localhost'
port = '5432'
database = 'customer_behaviour' 

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')  

#load data to postgresql
table_name = 'customer'
df.to_sql(table_name, engine, if_exists='replace', index=False)

print(f"Data has been successfully loaded into the '{table_name}' table in the PostgreSQL database.")

Data has been successfully loaded into the 'customer' table in the PostgreSQL database.


Questions to answer using sql: 

1. What is the total revenue generated by male vs female customers? 
    Female: 75191
    Male: 157890

2. Which customers used a discount but still spent more than the average purchase? 

3. which are the top 5 products with the highest average review rating? 

4. compare the average purchase amounts between standard and experess shipping

5. Do  subscribed customers spend more? compare average spend and total revenue between subscribers and non-subscribers

6. which 5 products have the highest percentage of purchases with discount applied? 

7. Segment customers into New, Returning, and Loyal based on their total number of previous purchases, and show the count of each segment

8. what are the top 3 most purchased products within each category? 

9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?
 
10. What is the revenue contribution of each group? 
